# 07 - Adstock Transformation

## Objective

Advertising rarely impacts sales only in the week it is shown. Customers remember ads,
search later, compare products, and purchase days or weeks afterwards.

Marketing Mix Models account for this delayed impact using **Adstock**.

In this notebook you will:

- Understand carryover effects
- Implement Geometric Adstock
- Implement Weibull Adstock
- Compare decay rates
- Transform all media channels
- Save an adstock-ready dataset



## Theory

For Geometric Adstock:

\[
Adstock_t = Spend_t + \theta \times Adstock_{t-1}
\]

Where:

- Spend = current media spend
- θ = decay factor (0–1)

Higher θ means advertising remains effective for longer.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT=Path.cwd()
DATA=ROOT/"data"/"processed"/"marketing_mix_outliers_treated.csv"

df=pd.read_csv(DATA,parse_dates=["Week"])

media_cols=[
"Google_Search","Google_Display","Meta","Instagram",
"YouTube","TV","Radio","Influencer","Affiliate","Email"
]

df.head()


## 1. Geometric Adstock

In [ ]:

def geometric_adstock(x, decay=0.5):
    result=np.zeros(len(x))
    result[0]=x.iloc[0]
    for i in range(1,len(x)):
        result[i]=x.iloc[i]+decay*result[i-1]
    return result


## 2. Visualize Different Decay Rates

In [ ]:

sample=df["Google_Search"]

decays=[0.2,0.5,0.7,0.9]

plt.figure(figsize=(14,5))
plt.plot(sample.values,label="Original",linewidth=2)

for d in decays:
    plt.plot(geometric_adstock(sample,d),label=f"Decay={d}")

plt.legend()
plt.title("Geometric Adstock Comparison")
plt.show()


## 3. Apply Adstock to Every Media Channel

In [ ]:

DECAY=0.6

for col in media_cols:
    df[col+"_Adstock"]=geometric_adstock(df[col],DECAY)

df.filter(regex="Adstock").head()


## 4. Weibull Adstock

In [ ]:

from scipy.stats import weibull_min

def weibull_adstock(series, shape=2.0, scale=5):
    weights=weibull_min.pdf(np.arange(1,21), shape, scale=scale)
    weights=weights/weights.sum()

    values=series.values
    output=np.zeros(len(values))

    for i in range(len(values)):
        total=0
        for j,w in enumerate(weights):
            idx=i-j
            if idx>=0:
                total+=values[idx]*w
        output[i]=total
    return output

df["Google_Search_Weibull"]=weibull_adstock(df["Google_Search"])
df[["Google_Search","Google_Search_Weibull"]].head()


## 5. Compare Original vs Geometric vs Weibull

In [ ]:

plt.figure(figsize=(14,5))

plt.plot(df["Google_Search"],label="Original",alpha=.5)
plt.plot(df["Google_Search_Adstock"],label="Geometric")
plt.plot(df["Google_Search_Weibull"],label="Weibull")

plt.legend()
plt.title("Adstock Comparison")
plt.show()


## 6. Correlation Improvement

In [ ]:

comparison=[]

for col in media_cols:
    raw=df[[col,"Sales"]].corr().iloc[0,1]
    ad=df[[col+"_Adstock","Sales"]].corr().iloc[0,1]
    comparison.append([col,raw,ad])

corr_df=pd.DataFrame(
comparison,
columns=["Channel","Raw Corr","Adstock Corr"]
)

display(corr_df.sort_values("Adstock Corr",ascending=False))


## 7. Save Dataset

In [ ]:

OUT=ROOT/"data"/"processed"/"marketing_mix_adstock.csv"
df.to_csv(OUT,index=False)
print("Saved:",OUT)


# Business Interpretation

### Why use Adstock?

Without Adstock, MMM assumes an advertisement influences sales only in the same week.

In reality:

- TV campaigns often influence purchases for several weeks.
- Brand campaigns have long memory.
- Search campaigns usually decay faster.
- Email campaigns often have short carryover.

Typical decay assumptions:

| Channel | Typical Decay |
|---------|--------------|
| Search | 0.2–0.5 |
| Meta | 0.3–0.6 |
| TV | 0.7–0.9 |
| Email | 0.1–0.3 |

These are starting points; in production, decay parameters are estimated or tuned.

---
